# Edge AI Challenge 2026 – Vòng loại: pipeline của đội

Detection: **YOLO P2/P3/P4 tự thiết kế, 191.145 tham số, train từ đầu**, distill từ một mô hình thầy cũng train từ đầu
trên dữ liệu BTC. Intent: **TextCNN** (Keras). Chạy từ trên xuống.

| `MODE` | Làm gì | Thời gian |
|---|---|---|
| `"infer"` | Dùng trọng số đã nộp trong `weights/` -> dự đoán test -> `submission.csv` | vài phút |
| `"train"` | Train lại từ đầu: thầy -> trò distill -> TextCNN -> `submission.csv` | khoảng 6 giờ trên T4 |

Dữ liệu: thêm dataset cuộc thi `egde-ai-challenge-2026-lan-2-qualifiers` (Kaggle: *Add Input*), hoặc đặt vào `data/`.
Cần Internet để `pip install ultralytics==8.4.199` (nếu môi trường chưa có).

## Cấu hình

In [ ]:
MODE = "infer"            # "infer" | "train"
QUICK_TEST = False        # True: 200 ảnh train, 2 epoch -> thử toàn bộ pipeline trong vài phút (KHÔNG dùng để nộp)
TRAIN_ON_ALL = False      # True: gộp val vào train (bài nộp 99,021 dùng False: train 4.800 ảnh, val 1.200 ảnh)

# Detection
TEACHER_EPOCHS = 200
STUDENT_EPOCHS = 300
STOP_EPOCH = 232          # bài nộp: trò dừng ở epoch 232/300 (máy hết RAM). 0 = chạy đủ STUDENT_EPOCHS
KD_CLS, KD_BOX = 1.0, 1.0 # hệ số loss distill: lớp (BCE theo xác suất của thầy) / box (KL phân bố DFL)
BATCH = 64
CLOSE_MOSAIC = 10
NMS_IOU, MAX_DET = 0.7, 100
SEED = 0
SPLIT_SEED = 42           # chia train/val 80/20

# Intent
INTENT_EPOCHS, INTENT_BATCH, MAX_LEN = 20, 64, 24
INTENT_SEED = 42

# Ràng buộc vòng loại
W_IN, H_IN = 320, 240
MAX_PARAMS, MAX_KB = 200_000, 1536
DATA_DIR = ""             # "" = tự tìm thư mục có classes.txt
REPO_URL = ""             # chỉ có file notebook (vd import lên Kaggle): link git của repo -> tự clone lấy weights/, yolo_cfg/

## Chuẩn bị: thư viện, đường dẫn

In [ ]:
import csv
import importlib.util
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.199"])

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
import ultralytics
from ultralytics import YOLO
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils.nms import non_max_suppression

if ultralytics.__version__ != "8.4.199":
    print(f"CHÚ Ý: đang dùng Ultralytics {ultralytics.__version__}, pipeline được kiểm tra với 8.4.199 "
          f"(KDLoss dùng API nội bộ của v8DetectionLoss)")

ON_KAGGLE = Path("/kaggle/input").exists()
DEVICE = 0 if torch.cuda.is_available() else "cpu"
WORKERS = 4 if ON_KAGGLE else 2
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def find_repo(clone=True):
    """Thư mục chứa yolo_cfg/ và weights/ (thư mục của notebook, bản clone, hoặc dataset trên Kaggle)."""
    roots = [Path.cwd(), Path.cwd().parent, Path("/kaggle/working"), Path("/kaggle/input"), Path("/content")]
    for root in roots:
        for pat in ("", "*", "*/*", "*/*/*"):
            for c in (root.glob(pat) if pat else [root]):
                if (c / "yolo_cfg" / "student_p2p4.yaml").exists():
                    return c.resolve()
    if clone and REPO_URL:
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", REPO_URL, str(Path.cwd() / "repo")])
        return find_repo(clone=False)
    raise FileNotFoundError("Không tìm thấy thư mục pipeline (có yolo_cfg/student_p2p4.yaml). Đặt REPO_URL "
                            "hoặc thêm thư mục pipeline làm dataset.")


def find_data_dir():
    if DATA_DIR:
        return Path(DATA_DIR)
    for root in (Path("/kaggle/input"), Path("/content/data"), REPO / "data", REPO.parent / "data", Path("data")):
        hits = sorted(root.rglob("classes.txt")) if root.exists() else []
        if hits:
            return hits[0].parent
    raise FileNotFoundError("Không tìm thấy dữ liệu cuộc thi (classes.txt). Đặt DATA_DIR.")


REPO = find_repo()
DATA = find_data_dir()
OUT = Path("/kaggle/working/out") if ON_KAGGLE else REPO / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
TRAIN_IMG, TRAIN_LBL, TEST_IMG = DATA / "train" / "images", DATA / "train" / "labels", DATA / "test" / "images"
CLASSES = [l.strip() for l in (DATA / "classes.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
INTENTS = ["COUNT", "EXIST", "LIST", "LOCATION", "OUT_OF_SCOPE"]
C = len(CLASSES)
print(f"MODE {MODE} | QUICK_TEST {QUICK_TEST} | TRAIN_ON_ALL {TRAIN_ON_ALL}")
print(f"Pipeline: {REPO}\nDữ liệu:  {DATA}\nKết quả:  {OUT}")
print(f"Ultralytics {ultralytics.__version__} | torch {torch.__version__} | GPU: "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ (chạy CPU)'}")
print("Lớp:", CLASSES)

## 1. Detection

### 1.1 Dữ liệu và hàm mAP@0.5
Chia train/val 80/20 theo `SPLIT_SEED`, chép sang cấu trúc thư mục của Ultralytics. mAP@0.5 = trung bình AP 7 lớp,
AP nội suy 101 điểm (cùng cách tính dùng để so mọi thí nghiệm của đội).

In [ ]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}


def list_images(folder):
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_EXTS)


def split_train_val(val_ratio=0.2, seed=SPLIT_SEED):
    imgs = list_images(TRAIN_IMG)
    random.Random(seed).shuffle(imgs)
    n_val = max(1, int(len(imgs) * val_ratio))
    return imgs[n_val:], imgs[:n_val]


def read_gt(img_path):
    """Nhãn YOLO (cls cx cy w h, 0..1) -> [(cls, [x1, y1, x2, y2] pixel)]."""
    out = []
    f = TRAIN_LBL / (img_path.stem + ".txt")
    for line in f.read_text().splitlines() if f.exists() else []:
        v = line.split()
        if len(v) == 5:
            c, x, y, w, h = int(float(v[0])), *map(float, v[1:])
            out.append((c, [(x - w / 2) * W_IN, (y - h / 2) * H_IN, (x + w / 2) * W_IN, (y + h / 2) * H_IN]))
    return out


def iou(box, boxes):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    ix1, iy1 = np.maximum(box[0], boxes[:, 0]), np.maximum(box[1], boxes[:, 1])
    ix2, iy2 = np.minimum(box[2], boxes[:, 2]), np.minimum(box[3], boxes[:, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area = (box[2] - box[0]) * (box[3] - box[1])
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / np.maximum(area + areas - inter, 1e-9)


def average_precision(tp, n_gt):
    if n_gt == 0:
        return None
    if len(tp) == 0:
        return 0.0
    tp = np.asarray(tp, dtype=float)
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    recall, precision = ctp / n_gt, ctp / (ctp + cfp)
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    ap = 0.0
    for r in np.linspace(0, 1, 101):
        idx = np.searchsorted(recall, r, side="left")
        ap += precision[idx] if idx < len(precision) else 0.0
    return ap / 101


def map50(preds, gts, iou_thr=0.5):
    per_class = {}
    for c in range(C):
        gt_c = {k: np.array([b for cc, b in v if cc == c], dtype=float).reshape(-1, 4) for k, v in gts.items()}
        n_gt = sum(len(v) for v in gt_c.values())
        if n_gt == 0:
            continue
        dets = [(k, p["score"], p["bbox"]) for k, ps in preds.items() if k in gts for p in ps if p["class_id"] == c]
        dets.sort(key=lambda d: -d[1])
        used = {k: np.zeros(len(v), bool) for k, v in gt_c.items()}
        tp = []
        for k, _, box in dets:
            g = gt_c.get(k)
            if g is None or len(g) == 0:
                tp.append(0)
                continue
            ious = iou(np.asarray(box, float), g)
            ious[used[k]] = -1
            j = int(ious.argmax())
            if ious[j] >= iou_thr:
                used[k][j] = True
                tp.append(1)
            else:
                tp.append(0)
        per_class[c] = average_precision(tp, n_gt)
    return float(np.mean(list(per_class.values()))), per_class


def build_dataset(train_paths, val_paths):
    """images/{train,val} + labels/{train,val} + data.yaml (tạo 1 lần, thầy và trò dùng chung)."""
    ds = (Path("/tmp") if ON_KAGGLE else OUT) / ("ds_quick" if QUICK_TEST else "ds_all" if TRAIN_ON_ALL else "ds_split")
    if (ds / "data.yaml").exists():
        return ds
    if ds.exists():
        shutil.rmtree(ds)
    for split, paths in (("train", train_paths), ("val", val_paths)):
        (ds / "images" / split).mkdir(parents=True)
        (ds / "labels" / split).mkdir(parents=True)
        for p in paths:
            shutil.copy2(p, ds / "images" / split / p.name)
            src = TRAIN_LBL / (p.stem + ".txt")
            (ds / "labels" / split / (p.stem + ".txt")).write_text(src.read_text() if src.exists() else "")
    cfg = {"path": str(ds), "train": "images/train", "val": "images/val", "names": dict(enumerate(CLASSES))}
    (ds / "data.yaml").write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding="utf-8")
    return ds


train_p, val_p = split_train_val()
if TRAIN_ON_ALL:
    train_p = train_p + val_p
if QUICK_TEST:
    train_p, val_p = train_p[:200], val_p[:100]
gts = {p.stem: read_gt(p) for p in val_p}
print(f"train {len(train_p)} ảnh | val {len(val_p)} ảnh{' (nằm trong train -> mAP val là GIẢ)' if TRAIN_ON_ALL else ''}")

### 1.2 Mô hình, distill, dự đoán

- `yolo_cfg/student_p2p4.yaml`: **trò (bài nộp)**, YOLO11 bỏ P5, đầu ra P2/4, P3/8, P4/16 (80×60 + 40×30 + 20×15 =
  6.300 anchor ở 320×240), giảm kích thước bằng SCDown, `reg_max` 4. Ảnh 320×240 đưa thẳng vào model (stride lớn
  nhất 16 nên không phải đệm).
- `yolo_cfg/teacher_p2p4.yaml`: **thầy**, 2,58 triệu tham số, cùng bố cục 6.300 anchor. Chỉ dùng lúc train trò,
  không nộp, không chạy lúc dự đoán.
- Distill: loss thường của Ultralytics + ở **mọi anchor**: BCE(logit lớp của trò, sigmoid(logit của thầy)) và
  KL(phân bố DFL của thầy || của trò) nhân độ tự tin của thầy; cả hai chia cho tổng độ tự tin của thầy.

In [ ]:
class KDLoss(v8DetectionLoss):
    def __init__(self, model, teacher):
        super().__init__(model)
        self.teacher = teacher

    def loss(self, preds, batch):
        total, items = super().loss(preds, batch)
        with torch.no_grad():
            tp = self.parse_output(self.teacher(batch["img"]))
        B = preds["scores"].shape[0]
        s_cls, t_cls = preds["scores"].float(), tp["scores"].float()
        t_p = t_cls.sigmoid()
        norm = t_p.sum().clamp(min=1)
        kd_cls = F.binary_cross_entropy_with_logits(s_cls, t_p, reduction="sum") / norm
        R = self.reg_max
        s_box = preds["boxes"].float().view(B, 4, R, -1)
        t_box = tp["boxes"].float().view(B, 4, R, -1)
        kl = (F.softmax(t_box, 2) * (F.log_softmax(t_box, 2) - F.log_softmax(s_box, 2))).sum(2)   # B, 4, A
        kd_box = (kl.sum(1) * t_p.max(1).values).sum() / norm
        return total + (KD_CLS * kd_cls + KD_BOX * kd_box) * B, items


def make_attach_kd(teacher_path):
    def attach_kd(trainer):
        teacher = YOLO(str(teacher_path)).model.to(trainer.device).float().eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        t_out = teacher(torch.zeros(1, 3, H_IN, W_IN, device=trainer.device))
        s_out = trainer.model.eval()(torch.zeros(1, 3, H_IN, W_IN, device=trainer.device))
        trainer.model.train()
        assert t_out[1]["boxes"].shape == s_out[1]["boxes"].shape, "Thầy và trò khác bố cục đầu ra"
        trainer.model.criterion = KDLoss(trainer.model, teacher)
        print(f"Distill từ {teacher_path} ({sum(p.numel() for p in teacher.parameters()):,} tham số), "
              f"KD_CLS {KD_CLS}, KD_BOX {KD_BOX}", flush=True)
    return attach_kd


def train(run, cfg, epochs, ds, teacher=None, stop_epoch=0, is_teacher=False):
    """Dựng model TỪ YAML (trọng số ngẫu nhiên, không pretrained) rồi train. Trả về thư mục weights/."""
    model = YOLO(str(REPO / "yolo_cfg" / cfg))
    n = sum(p.numel() for p in model.model.parameters())
    print(f"[{run}] {cfg}: {n:,} tham số{' (thầy, không nộp)' if is_teacher else ''}", flush=True)
    assert is_teacher or n <= MAX_PARAMS, "VƯỢT trần tham số"
    if teacher:
        model.add_callback("on_train_start", make_attach_kd(teacher))
    if stop_epoch:
        def stop(trainer):
            if trainer.epoch + 1 >= stop_epoch:
                trainer.stop = True
        model.add_callback("on_fit_epoch_end", stop)
    t0 = time.time()
    model.train(data=str(ds / "data.yaml"), epochs=epochs, imgsz=W_IN, batch=BATCH, workers=WORKERS,
                device=DEVICE, pretrained=False, seed=SEED, deterministic=False, project=str(OUT), name=run,
                exist_ok=True, plots=False, patience=0, close_mosaic=CLOSE_MOSAIC, verbose=False)
    print(f"[{run}] train xong sau {(time.time() - t0) / 60:.1f} phút", flush=True)
    return Path(model.trainer.save_dir) / "weights"


@torch.no_grad()
def predict(weights, paths):
    """Ảnh gốc 320x240 đưa THẲNG vào model (1 lần/ảnh, không TTA), NMS -> box pixel ảnh gốc."""
    model = YOLO(str(weights)).model.to(DEVICE).float().eval()
    out = {}
    for k in range(0, len(paths), 64):
        chunk = paths[k:k + 64]
        imgs = []
        for p in chunk:
            im = cv2.imread(str(p))
            assert im.shape[:2] == (H_IN, W_IN), (p, im.shape)
            imgs.append(im[:, :, ::-1].transpose(2, 0, 1))   # BGR -> RGB, HWC -> CHW
        x = torch.from_numpy(np.ascontiguousarray(np.stack(imgs))).to(DEVICE).float() / 255
        y = model(x)
        y = y[0] if isinstance(y, (list, tuple)) else y
        dets = non_max_suppression(y, conf_thres=0.001, iou_thres=NMS_IOU, max_det=MAX_DET)
        for p, d in zip(chunk, dets):
            d = d.cpu().numpy()
            out[p.stem] = [{"class_id": int(c), "score": round(float(s), 4),
                            "bbox": [round(float(min(max(x1, 0), W_IN)), 1), round(float(min(max(y1, 0), H_IN)), 1),
                                     round(float(min(max(x2, 0), W_IN)), 1), round(float(min(max(y2, 0), H_IN)), 1)]}
                           for x1, y1, x2, y2, s, c in d[:, :6]]
    return out


def eval_val(weights, name):
    mp, per = map50(predict(weights, val_p), gts)
    print(f"mAP50 val {name}: {mp:.4f} | " + " ".join(f"{CLASSES[c]} {a:.3f}" for c, a in per.items()), flush=True)
    return mp

### 1.3 Train (chỉ khi `MODE = "train"`): thầy -> trò distill
Bài nộp: thầy 200 epoch, chọn `best.pt`; trò distill dừng ở epoch 232/300, dùng `last.pt` (mAP val 0,9819).

In [ ]:
if MODE == "train":
    t_ep, s_ep = (2, 2) if QUICK_TEST else (TEACHER_EPOCHS, STUDENT_EPOCHS)
    ds = build_dataset(train_p, val_p)
    t_dir = train("teacher", "teacher_p2p4.yaml", t_ep, ds, is_teacher=True)
    if not TRAIN_ON_ALL:
        eval_val(t_dir / "best.pt", "thầy best.pt")
    TEACHER_W = t_dir / ("last.pt" if TRAIN_ON_ALL else "best.pt")
    s_dir = train("student_kd", "student_p2p4.yaml", s_ep, ds, teacher=TEACHER_W,
                  stop_epoch=0 if QUICK_TEST else STOP_EPOCH)
    if TRAIN_ON_ALL:
        STUDENT_W = s_dir / "last.pt"
    else:
        scores = {w: eval_val(s_dir / w, f"trò {w}") for w in ("best.pt", "last.pt")}
        STUDENT_W = s_dir / max(scores, key=scores.get)
else:
    STUDENT_W = REPO / "weights" / "yolo_kd_student.pt"
print("Trọng số dùng để dự đoán:", STUDENT_W)

### 1.4 Kiểm tra ràng buộc, mAP val, dự đoán test

In [ ]:
final = YOLO(str(STUDENT_W)).model.float().eval()
n_params = sum(p.numel() for p in final.parameters())
kb_fp32 = n_params * 4 / 1024
with torch.no_grad():
    y = final.to(DEVICE)(torch.zeros(1, 3, H_IN, W_IN, device=DEVICE))
y = y[0] if isinstance(y, (list, tuple)) else y
n_anchor = (W_IN // 4) * (H_IN // 4) + (W_IN // 8) * (H_IN // 8) + (W_IN // 16) * (H_IN // 16)
print(f"Số tham số: {n_params:,} (<= {MAX_PARAMS:,}) {'OK' if n_params <= MAX_PARAMS else 'VƯỢT'}")
print(f"Trọng số float32: {kb_fp32:.0f} KB (<= {MAX_KB} KB) {'OK' if kb_fp32 <= MAX_KB else 'VƯỢT'}")
print(f"Đầu vào 1x3x{H_IN}x{W_IN} -> đầu ra {tuple(y.shape)} (4 box + {C} lớp, {n_anchor} anchor P2/P3/P4)")
assert n_params <= MAX_PARAMS and kb_fp32 <= MAX_KB and tuple(y.shape) == (1, 4 + C, n_anchor)

if not TRAIN_ON_ALL:
    VAL_MAP = eval_val(STUDENT_W, STUDENT_W.name)

test_p = list_images(TEST_IMG)
t0 = time.time()
det = predict(STUDENT_W, test_p)
(OUT / "det_test.json").write_text(json.dumps(det, separators=(",", ":")), encoding="utf-8")
print(f"Test: {len(det)} ảnh, {sum(map(len, det.values()))} box, {time.time() - t0:.0f}s")

## 2. Intent: TextCNN (Keras)

Chuẩn hóa câu (chữ thường, bỏ ký tự ngoài `a-z0-9`) -> tách từ theo dấu cách -> vocab -> 24 token ->
Embedding 32 + Conv1D (kernel 2, 3, 4) + GlobalMaxPool -> Dense 5 lớp.

Lúc **train** có sinh thêm câu: câu COUNT/EXIST/LOCATION có tên vật thuộc 7 lớp -> thêm 1 câu đổi sang tên khác trong
7 lớp (giữ nhãn) và 1 câu đổi sang vật ngoài 7 lớp (nhãn OUT_OF_SCOPE). Lúc **dự đoán** chỉ chạy TextCNN, không dùng
từ điển hay luật nào.

In [ ]:
import tensorflow as tf

tf.config.set_visible_devices([], "GPU")   # TextCNN rất nhỏ: chạy CPU, không tranh GPU với PyTorch
L = tf.keras.layers
label2id = {l: i for i, l in enumerate(INTENTS)}


def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


class Vocab:
    def __init__(self, texts=None, itos=None):
        self.itos = itos or ["<pad>", "<oov>"] + sorted({w for t in texts for w in normalize(t).split()})
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, text, max_len=MAX_LEN):
        ids = [self.stoi.get(w, 1) for w in normalize(text).split()][:max_len]
        return ids + [0] * (max_len - len(ids))


IN_SCOPE_NAMES = ["chai", "chai nuoc", "chai nhua", "binh nuoc", "coc", "coc uong nuoc", "ly", "tach",
                  "dien thoai", "dien thoai di dong", "smartphone", "sach", "cuon sach", "quyen sach",
                  "chuot", "chuot may tinh", "con chuot may tinh", "hop", "cai hop", "hop giay", "thung giay",
                  "oto do choi", "o to do choi", "xe oto do choi", "xe do choi"]
OUT_SCOPE_NAMES = ["binh hoa", "cai thia", "den ban", "tai nghe", "cai vi", "cay but", "cai remote", "cai dia",
                   "banh mi", "con meo", "cuon giay ve sinh", "qua chuoi", "cai mu", "dong xu", "laptop",
                   "cuc tay", "doi giay", "cai thuoc ke", "con cho", "cai bat", "cai keo", "kinh mat", "cai quat",
                   "chia khoa", "con gau bong", "o cam dien", "may tinh bang", "cay nen", "qua bong", "cai o",
                   "ban phim", "dong ho", "cai ghe", "em be", "chau cay", "con chim", "qua tao"]
_IN_SORTED = sorted(set(IN_SCOPE_NAMES), key=len, reverse=True)
_IN_RE = re.compile(r"\b(" + "|".join(re.escape(n) for n in _IN_SORTED) + r")\b")


def augment_with_dictionary(texts, labels, seed=0):
    rng = random.Random(seed)
    out_t, out_l = [], []
    for t, l in zip(texts, labels):
        t = normalize(t)
        out_t.append(t)
        out_l.append(l)
        if l not in ("COUNT", "EXIST", "LOCATION") or not _IN_RE.search(t):
            continue
        new = _IN_RE.sub(lambda m: rng.choice(_IN_SORTED), t, count=1)
        if new != t:
            out_t.append(new)
            out_l.append(l)
        out_t.append(_IN_RE.sub(lambda m: rng.choice(OUT_SCOPE_NAMES), t, count=1))
        out_l.append("OUT_OF_SCOPE")
    return out_t, out_l


def build_textcnn(vocab_size):
    tin = L.Input((MAX_LEN,), dtype="int32", name="tokens")
    e = L.Embedding(vocab_size, 32)(tin)
    br = [L.GlobalMaxPooling1D()(L.Conv1D(32, k, activation="relu", padding="same")(e)) for k in (2, 3, 4)]
    h = L.Dropout(0.3)(L.Concatenate()(br))
    m = tf.keras.Model(tin, L.Dense(len(INTENTS), activation="softmax")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(3e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def train_intent(texts, labels, epochs):
    t, l = augment_with_dictionary(texts, labels, seed=INTENT_SEED)
    vocab = Vocab(t)
    X = np.array([vocab.encode(q) for q in t], np.int32)
    y = np.array([label2id[v] for v in l], np.int32)
    present = np.unique(y)
    cw = dict(zip(present.tolist(), (len(y) / (len(present) * np.bincount(y)[present])).tolist()))
    m = build_textcnn(len(vocab))
    m.fit(X, y, epochs=epochs, batch_size=INTENT_BATCH, class_weight=cw, verbose=0, shuffle=True)
    return m, vocab, len(X)


itrain = pd.read_csv(DATA / "intent_train.csv")
itest = pd.read_csv(DATA / "intent_test.csv")
if MODE == "train":
    tf.keras.utils.set_random_seed(INTENT_SEED)
    ep = 2 if QUICK_TEST else INTENT_EPOCHS
    idx = np.random.default_rng(INTENT_SEED).permutation(len(itrain))
    va, tr = idx[:len(itrain) // 5], idx[len(itrain) // 5:]
    m_chk, v_chk, _ = train_intent(itrain.question.iloc[tr], itrain.intent.iloc[tr], ep)   # sinh câu chỉ trên 80%
    Xv = np.array([v_chk.encode(q) for q in itrain.question.iloc[va]], np.int32)
    acc = (m_chk.predict(Xv, verbose=0).argmax(1) == itrain.intent.iloc[va].map(label2id).values).mean()
    print(f"Holdout 20% câu gốc: accuracy {acc:.4f}")
    imodel, vocab, n_aug = train_intent(itrain.question, itrain.intent, ep)
    print(f"Mô hình cuối: {len(itrain)} câu gốc -> {n_aug} câu train")
    imodel.save(OUT / "intent_textcnn.keras")
    (OUT / "intent_vocab.txt").write_text("\n".join(vocab.itos), encoding="utf-8")
else:
    imodel = tf.keras.models.load_model(REPO / "weights" / "intent_textcnn.keras")
    vocab = Vocab(itos=(REPO / "weights" / "intent_vocab.txt").read_text(encoding="utf-8").split("\n"))
print(f"TextCNN: {imodel.count_params():,} tham số | vocab {len(vocab)}")
Xt = np.array([vocab.encode(q) for q in itest.question], np.int32)
itest["intent"] = [INTENTS[i] for i in imodel.predict(Xt, verbose=0).argmax(1)]
intent = dict(zip(itest.question_id.astype(str), itest.intent))
print("Phân bố dự đoán test:", itest.intent.value_counts().to_dict())

## 3. Ghép `submission.csv` và kiểm tra

In [ ]:
sample = pd.read_csv(DATA / "sample_submission.csv", dtype=str)


def clean_boxes(boxes):
    good = []
    for b in boxes:
        x1, y1, x2, y2 = (float(v) for v in b["bbox"])
        s, c = float(b["score"]), int(b["class_id"])
        if not all(math.isfinite(v) for v in (x1, y1, x2, y2, s)) or x2 <= x1 or y2 <= y1 or not 0 <= c < C:
            continue
        good.append({"class_id": c, "score": round(min(max(s, 0.0), 1.0), 4),
                     "bbox": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]})
    good.sort(key=lambda b: -b["score"])
    return good[:MAX_DET]


rows, missing = [], {"DETECTION": 0, "INTENT": 0}
for task, sid in zip(sample.task_type, sample.sample_id):
    if task == "DETECTION":
        missing[task] += sid not in det
        rows.append((task, sid, json.dumps(clean_boxes(det.get(sid, [])), separators=(",", ":"))))
    else:
        missing[task] += sid not in intent
        rows.append((task, sid, intent.get(sid, "COUNT")))

SUB = (Path("/kaggle/working") if ON_KAGGLE else OUT) / "submission.csv"
with open(SUB, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["task_type", "sample_id", "prediction"])
    w.writerows(rows)

check = pd.read_csv(SUB, dtype=str, keep_default_na=False)
errors = [m for m, bad in [
    (f"số dòng {len(check)} != {len(sample)}", len(check) != len(sample)),
    ("thứ tự sample_id khác file mẫu", list(check.sample_id) != list(sample.sample_id)),
    (f"thiếu dự đoán {missing}", any(missing.values())),
    ("QUICK_TEST = True: bản chạy thử, không nộp", QUICK_TEST)] if bad]
print(f"Detection {(check.task_type == 'DETECTION').sum()} ảnh | Intent {(check.task_type == 'INTENT').sum()} câu")
print("LỖI:" if errors else f"OK - file hợp lệ: {SUB}", *errors, sep="\n  - " if errors else "")

## Tóm tắt

In [ ]:
print(f"""MODE {MODE}
Detection: {STUDENT_W.name} | {n_params:,} tham số | {kb_fp32:.0f} KB float32 | đầu vào {W_IN}x{H_IN}
mAP50 val: {f'{VAL_MAP:.4f}' if not TRAIN_ON_ALL else 'không đo (TRAIN_ON_ALL)'}
Intent: TextCNN {imodel.count_params():,} tham số
Kết quả: {SUB}""")